In [2]:
!apt-get install -y nvidia-cuda-toolkit


Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
nvidia-cuda-toolkit is already the newest version (11.5.1-1ubuntu1).
0 upgraded, 0 newly installed, 0 to remove and 51 not upgraded.


In [1]:

!nvcc --version


nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


In [2]:

!nvidia-smi

Thu May 14 00:54:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [1]:
%%writefile cuda_assignment.cu

#include <iostream>
#include <cuda.h>
#include <chrono>

using namespace std;
using namespace std::chrono;

// ================= VECTOR ADDITION =================

__global__ void vectorAdd(int *A, int *B, int *C, int n) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    if (i < n) {
        C[i] = A[i] + B[i];
    }
}

void sequentialVectorAdd(int *A, int *B, int *C, int n) {
    for (int i = 0; i < n; i++) {
        C[i] = A[i] + B[i];
    }
}

// ================= MATRIX MULTIPLICATION =================

__global__ void matrixMul(int *A, int *B, int *C, int N) {
    int row = blockIdx.y * blockDim.y + threadIdx.y;
    int col = blockIdx.x * blockDim.x + threadIdx.x;

    if (row < N && col < N) {
        int sum = 0;

        for (int k = 0; k < N; k++) {
            sum += A[row * N + k] * B[k * N + col];
        }

        C[row * N + col] = sum;
    }
}

void sequentialMatrixMul(int *A, int *B, int *C, int N) {
    for (int i = 0; i < N; i++) {
        for (int j = 0; j < N; j++) {
            int sum = 0;

            for (int k = 0; k < N; k++) {
                sum += A[i * N + k] * B[k * N + j];
            }

            C[i * N + j] = sum;
        }
    }
}

// ================= MAIN =================

int main() {

    // ---------------- VECTOR ADDITION ----------------
    int n = 30000000;
    int size = n * sizeof(int);

    int *h_A = new int[n];
    int *h_B = new int[n];
    int *h_C_seq = new int[n];
    int *h_C_par = new int[n];

    for (int i = 0; i < n; i++) {
        h_A[i] = i;
        h_B[i] = i * 2;
    }

    int *d_A, *d_B, *d_C;

    cudaMalloc((void**)&d_A, size);
    cudaMalloc((void**)&d_B, size);
    cudaMalloc((void**)&d_C, size);

    cudaMemcpy(d_A, h_A, size, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, size, cudaMemcpyHostToDevice);

    auto start = high_resolution_clock::now();
    sequentialVectorAdd(h_A, h_B, h_C_seq, n);
    auto stop = high_resolution_clock::now();

    cout << "Sequential Vector Addition Time: "
         << duration_cast<milliseconds>(stop - start).count()
         << " ms\n";

    start = high_resolution_clock::now();

    int threadsPerBlock = 256;
    int blocksPerGrid = (n + threadsPerBlock - 1) / threadsPerBlock;

    vectorAdd<<<blocksPerGrid, threadsPerBlock>>>(d_A, d_B, d_C, n);
    cudaDeviceSynchronize();

    stop = high_resolution_clock::now();

    cudaMemcpy(h_C_par, d_C, size, cudaMemcpyDeviceToHost);

    cout << "Parallel Vector Addition Time (CUDA): "
         << duration_cast<milliseconds>(stop - start).count()
         << " ms\n";

    // ---------------- MATRIX MULTIPLICATION ----------------
    int N = 512;
    int matrixSize = N * N * sizeof(int);

    int *h_MA = new int[N * N];
    int *h_MB = new int[N * N];
    int *h_MC_seq = new int[N * N];
    int *h_MC_par = new int[N * N];

    for (int i = 0; i < N * N; i++) {
        h_MA[i] = 1;
        h_MB[i] = 2;
    }

    int *d_MA, *d_MB, *d_MC;

    cudaMalloc((void**)&d_MA, matrixSize);
    cudaMalloc((void**)&d_MB, matrixSize);
    cudaMalloc((void**)&d_MC, matrixSize);

    cudaMemcpy(d_MA, h_MA, matrixSize, cudaMemcpyHostToDevice);
    cudaMemcpy(d_MB, h_MB, matrixSize, cudaMemcpyHostToDevice);

    start = high_resolution_clock::now();
    sequentialMatrixMul(h_MA, h_MB, h_MC_seq, N);
    stop = high_resolution_clock::now();

    cout << "Sequential Matrix Multiplication Time: "
         << duration_cast<milliseconds>(stop - start).count()
         << " ms\n";

    dim3 threads(16, 16);
    dim3 blocks((N + threads.x - 1) / threads.x,
                (N + threads.y - 1) / threads.y);

    start = high_resolution_clock::now();

    matrixMul<<<blocks, threads>>>(d_MA, d_MB, d_MC, N);
    cudaDeviceSynchronize();

    stop = high_resolution_clock::now();

    cudaMemcpy(h_MC_par, d_MC, matrixSize, cudaMemcpyDeviceToHost);

    cout << "Parallel Matrix Multiplication Time (CUDA): "
         << duration_cast<milliseconds>(stop - start).count()
         << " ms\n";

    return 0;
}


Writing cuda_assignment.cu


In [2]:
!nvcc cuda_assignment.cu -o cuda_assignment

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [3]:
!./cuda_assignment

Sequential Vector Addition Time: 140 ms
Parallel Vector Addition Time (CUDA): 109 ms
Sequential Matrix Multiplication Time: 615 ms
Parallel Matrix Multiplication Time (CUDA): 0 ms
